In [1]:
import sys

PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)


from mhcnames import normalize_allele_name
import numpy as np
import pandas as pd
from config import RESULT_COLUMNS, CLASSES_TO_USE
CLASSES_TO_USE

['I', 'II']

## Read the data

In [2]:
DATA_PATH = '/home/denklewer/Data/schreiber_full/'

In [3]:
per_class_df = {class_to_use :  pd.read_csv("{}A20CP_10E6Cell-sorted-predictions-class{}.csv".format(DATA_PATH, class_to_use), sep=';') for class_to_use in CLASSES_TO_USE}

In [4]:
per_allele_mt_df = dict()
per_allele_wt_df = dict()
per_allele_total_df = dict()

In [6]:
ALLELES

['H-2-Kd', 'H-2-Ld', 'H-2-Dd']

In [5]:
ALLELES = list()

for class_to_use in CLASSES_TO_USE:
    df = per_class_df[class_to_use]
    ALLELES_FOR_CLASS = df['allele'].unique()
    ALLELES.extend(ALLELES_FOR_CLASS)
    
    for allele_name in ALLELES_FOR_CLASS:
        print(allele_name)
        allele_df = df.loc[df['allele'] == allele_name].copy()

        # total df
        allele_df[RESULT_COLUMNS['id']] =  'seq' + allele_df.index.astype(str).values
        allele_df = allele_df.rename(columns= {'peptide_length': 'length', 'peptide_mut':'peptide_mt'})

        
        df_wt = allele_df[[RESULT_COLUMNS['id'], 'allele', 'peptide_wt', 'length']].copy()
        df_wt = df_wt.rename(columns={'peptide_wt':'peptide'})
        
        
        df_mt = allele_df[[RESULT_COLUMNS['id'], 'allele', 'peptide_mt', 'length']].copy()
        df_mt = df_mt.rename(columns={'peptide_mt':'peptide'})
        print(len(df_mt), len(df_wt))
        
            
        per_allele_mt_df[allele_name] = df_mt
        per_allele_wt_df[allele_name] = df_wt
        per_allele_total_df[allele_name] = allele_df

        
#stack all the dfs
list_of_df = per_allele_total_df.values()
total_df = pd.concat(list_of_df)
        
        


H-2-Kd
8738 8738
H-2-Ld
8738 8738
H-2-Dd
8738 8738


In [39]:
len(total_df), len(df_wt), len(df_mt)

(130396, 52091, 52091)

In [47]:
ALLELES

['H-2-Kd', 'H-2-Ld', 'H-2-Dd', 'H-2-IEd', 'H-2-IAd']

# Write to files

In [44]:
import os
if not os.path.exists(DATA_PATH):
    os.makedirs(DATA_PATH)

In [45]:
# for decoy_key in DECOY_PREFIXES:
for allele_name in ALLELES:
    df_mt = per_allele_mt_df[allele_name]
    df_wt = per_allele_wt_df[allele_name]
    allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    df_mt.to_csv("{}schreiber_data_full_{}_mt.csv".format(DATA_PATH,allele_name),sep=";", index=False)
    df_wt.to_csv("{}schreiber_data_full_{}_wt.csv".format(DATA_PATH,allele_name),sep=";", index=False)
    total_df.to_csv("{}schreiber_total_data_full.csv".format(DATA_PATH),sep=";", index=False)